<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase21_markowitz_shrinkage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 21: Markowitz completo, rendimientos esperados y shrinkage

Corre con *Entorno de ejecución > Reiniciar sesión y ejecutar todo*.

**Antes de publicar resultados de las partes 6 y 7:** llena la tabla `tasas` con las tasas históricas reales y cambia `TASAS_DE_RELLENO` a `False`. Con las tasas de relleno, esas dos partes solo sirven para probar que el código corre.

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf

# 1. Dos activos a mano: portafolio de máximo Sharpe (tasa libre de 0)
mu = np.array([0.04, 0.08])
vol = np.array([0.10, 0.20])
rho = 0.3
cov = np.outer(vol, vol) * np.array([[1, rho], [rho, 1]])
x = np.linalg.solve(cov, mu)
w = x / x.sum()
ret_p = float(w @ mu)
vol_p = float((w @ cov @ w)**0.5)
print("Sharpe de cada activo solo:", (mu / vol).round(3))
print("Pesos de máximo Sharpe:", w.round(3), "| rendimiento:", round(ret_p, 4),
      "| volatilidad:", round(vol_p, 4), "| Sharpe:", round(ret_p / vol_p, 3))
print("Frontera (peso en el activo 1, rendimiento, volatilidad, Sharpe):")
for a in np.linspace(0, 1, 6):
    wa = np.array([a, 1 - a])
    r, v = float(wa @ mu), float((wa @ cov @ wa)**0.5)
    print("  ", round(a, 1), round(r, 4), round(v, 4), round(r / v, 3))

# 2. Sensibilidad: dos activos casi idénticos y una diferencia de 0.5 puntos en el rendimiento esperado
cov2 = 0.01 * np.array([[1, 0.9], [0.9, 1]])
for m in ([0.05, 0.055], [0.055, 0.05]):
    x2 = np.linalg.solve(cov2, np.array(m))
    print("Rendimientos esperados", m, "-> pesos:", (x2 / x2.sum()).round(3))
print("Error estándar del rendimiento promedio con 4 años y volatilidad de 10%:", round(0.10 / np.sqrt(4), 3))

Sharpe de cada activo solo: [0.4 0.4]
Pesos de máximo Sharpe: [0.667 0.333] | rendimiento: 0.0533 | volatilidad: 0.1075 | Sharpe: 0.496
Frontera (peso en el activo 1, rendimiento, volatilidad, Sharpe):
   0.0 0.08 0.2 0.4
   0.2 0.072 0.1671 0.431
   0.4 0.064 0.1374 0.466
   0.6 0.056 0.1135 0.493
   0.8 0.048 0.0996 0.482
   1.0 0.04 0.1 0.4
Rendimientos esperados [0.05, 0.055] -> pesos: [0.048 0.952]
Rendimientos esperados [0.055, 0.05] -> pesos: [0.952 0.048]
Error estándar del rendimiento promedio con 4 años y volatilidad de 10%: 0.05


In [3]:
# 3. Funciones (entradas anualizadas)
def min_var_solo_largo(cov, tope=1.0):
    n = len(cov)
    c = cov / np.mean(np.diag(cov))
    res = minimize(lambda w: w @ c @ w, np.ones(n) / n, method="SLSQP", bounds=[(0, tope)] * n,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   options={"ftol": 1e-14, "maxiter": 500})
    w = res.x.copy()
    w[np.abs(w) < 1e-6] = 0.0
    return w / w.sum()

def max_sharpe_solo_largo(mu, cov, tope=1.0, avisar=True):
    n = len(mu)
    if mu.max() <= 0:
        if avisar:
            print("Aviso: ningún rendimiento esperado es positivo; el máximo Sharpe solo largo no tiene sentido. Se usa mínima varianza.")
        return min_var_solo_largo(cov, tope)
    res = minimize(lambda w: -(w @ mu) / np.sqrt(w @ cov @ w), np.ones(n) / n, method="SLSQP",
                   bounds=[(0, tope)] * n,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   options={"ftol": 1e-14, "maxiter": 500})
    w = res.x.copy()
    w[np.abs(w) < 1e-6] = 0.0
    return w / w.sum()

def encoger_mu(mu_hat, lam):
    return lam * mu_hat + (1 - lam) * mu_hat.mean()   # lam = 1 no encoge, lam = 0 todos iguales

In [4]:
# 4. Simulación: ¿qué pasa con el error de estimación? (se conocen los parámetros verdaderos)
vols = np.array([0.08, 0.10, 0.12, 0.10, 0.15, 0.09])
corr = np.full((6, 6), 0.5)
np.fill_diagonal(corr, 1)
cov_v = np.outer(vols, vols) * corr
escenarios = {
    "A: activos con Sharpe verdadero parecido": vols * np.array([0.25, 0.30, 0.33, 0.30, 0.33, 0.22]),
    "B: activos con Sharpe verdadero muy distinto": vols * np.array([0.0, 0.1, 0.3, 0.5, 0.7, 0.9]),
}
nombres = ["Pesos iguales", "Mínima varianza", "Máximo Sharpe con estimaciones", "Máximo Sharpe encogido"]
for etiqueta, mu_v in escenarios.items():
    rng = np.random.default_rng(123)
    def sharpe_verdadero(w):
        return float(w @ mu_v / np.sqrt(w @ cov_v @ w))
    res_sim = {k: [] for k in nombres}
    for _ in range(300):
        X = rng.multivariate_normal(mu_v / 252, cov_v / 252, size=1000)       # 1000 días, unos 4 años
        mu_hat, cov_hat = X.mean(axis=0) * 252, np.cov(X.T) * 252
        cov_lw = LedoitWolf().fit(X).covariance_ * 252
        res_sim["Pesos iguales"].append(sharpe_verdadero(np.ones(6) / 6))
        res_sim["Mínima varianza"].append(sharpe_verdadero(min_var_solo_largo(cov_hat)))
        res_sim["Máximo Sharpe con estimaciones"].append(sharpe_verdadero(max_sharpe_solo_largo(mu_hat, cov_hat, avisar=False)))
        res_sim["Máximo Sharpe encogido"].append(sharpe_verdadero(max_sharpe_solo_largo(encoger_mu(mu_hat, 0.25), cov_lw, avisar=False)))
    print("\nEscenario", etiqueta)
    print("  Sharpe verdadero de la mejor cartera posible:", round(sharpe_verdadero(max_sharpe_solo_largo(mu_v, cov_v)), 3))
    for k in nombres:
        v = np.array(res_sim[k])
        print(f"  {k}: promedio {v.mean():.3f} | percentil 10: {np.percentile(v, 10):.3f}")


Escenario A: activos con Sharpe verdadero parecido
  Sharpe verdadero de la mejor cartera posible: 0.401
  Pesos iguales: promedio 0.385 | percentil 10: 0.385
  Mínima varianza: promedio 0.317 | percentil 10: 0.310
  Máximo Sharpe con estimaciones: promedio 0.324 | percentil 10: 0.265
  Máximo Sharpe encogido: promedio 0.324 | percentil 10: 0.296

Escenario B: activos con Sharpe verdadero muy distinto
  Sharpe verdadero de la mejor cartera posible: 0.945
  Pesos iguales: promedio 0.575 | percentil 10: 0.575
  Mínima varianza: promedio 0.414 | percentil 10: 0.371
  Máximo Sharpe con estimaciones: promedio 0.798 | percentil 10: 0.606
  Máximo Sharpe encogido: promedio 0.693 | percentil 10: 0.458


In [5]:
# 5. Datos reales: seis pares contra el dólar
pares = {"USD/MXN": ("MXN=X", False), "USD/JPY": ("JPY=X", False), "USD/EUR": ("EURUSD=X", True),
         "USD/GBP": ("GBPUSD=X", True), "USD/CAD": ("CAD=X", False), "USD/BRL": ("BRL=X", False)}
precios = {}
for nombre, (ticker, invertir) in pares.items():
    p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
    precios[nombre] = 1 / p if invertir else p
rend = pd.DataFrame(precios).dropna().pct_change().dropna()
corte = int(len(rend) * 0.7)
entren, prueba = rend.iloc[:corte], rend.iloc[corte:]
print("\nDías:", len(rend), "| del", rend.index[0].date(), "al", rend.index[-1].date(),
      "| entrenamiento:", len(entren), "| prueba:", len(prueba))

mu_e = entren.mean().values * 252
cov_e = entren.cov().values * 252
lw = LedoitWolf().fit(entren.values)
cov_lw_e = lw.covariance_ * 252
print("Intensidad del encogimiento de Ledoit Wolf:", round(float(lw.shrinkage_), 3))
print("Rendimiento anual esperado estimado (solo precio, sin carry) por par:")
print((pd.Series(mu_e, index=rend.columns) * 100).round(2).to_string())

carteras = {
    "Pesos iguales": np.ones(6) / 6,
    "Mínima varianza (cov. encogida)": min_var_solo_largo(cov_lw_e),
    "Máximo Sharpe (sin encoger)": max_sharpe_solo_largo(mu_e, cov_e),
    "Máximo Sharpe (encogido)": max_sharpe_solo_largo(encoger_mu(mu_e, 0.25), cov_lw_e),
}
pesos = pd.DataFrame(carteras, index=rend.columns)
print(pesos.round(3).to_string())
filas = []
for nombre, w in carteras.items():
    ser_in, ser_out = entren.values @ w, prueba.values @ w
    filas.append({"cartera": nombre,
                  "Sharpe_dentro": ser_in.mean() / ser_in.std() * np.sqrt(252),
                  "rend_fuera_%": ser_out.mean() * 252 * 100,
                  "vol_fuera_%": ser_out.std() * np.sqrt(252) * 100,
                  "Sharpe_fuera": ser_out.mean() / ser_out.std() * np.sqrt(252)})
print(pd.DataFrame(filas).round(3).to_string(index=False))

/tmp/ipykernel_440/2019623763.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
/tmp/ipykernel_440/2019623763.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
/tmp/ipykernel_440/2019623763.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
/tmp/ipykernel_440/2019623763.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
/tmp/ipykernel_440/2019623763.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2


Días: 1570 | del 2020-10-08 al 2026-10-21 | entrenamiento: 1099 | prueba: 471
Intensidad del encogimiento de Ledoit Wolf: 0.012
Rendimiento anual esperado estimado (solo precio, sin carry) por par:
USD/MXN   -1.04
USD/JPY    9.55
USD/EUR    3.03
USD/GBP    1.00
USD/CAD    1.97
USD/BRL    3.48
         Pesos iguales  Mínima varianza (cov. encogida)  Máximo Sharpe (sin encoger)  Máximo Sharpe (encogido)
USD/MXN          0.167                            0.077                        0.000                     0.000
USD/JPY          0.167                            0.208                        0.935                     0.439
USD/EUR          0.167                            0.239                        0.000                     0.160
USD/GBP          0.167                            0.000                        0.000                     0.000
USD/CAD          0.167                            0.476                        0.007                     0.374
USD/BRL          0.167                 

/tmp/ipykernel_440/2019623763.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()


In [6]:
# 6. Carry: una posición larga en dólares contra otra moneda gana o paga el diferencial de tasas
mapa_moneda = {"USD/MXN": "MXN", "USD/JPY": "JPY", "USD/EUR": "EUR",
               "USD/GBP": "GBP", "USD/CAD": "CAD", "USD/BRL": "BRL"}
# EDITA AQUÍ: tasas de política monetaria en % anual, una fila por cada fecha en que cambió la tasa de cualquiera de los bancos.
# La primera fila debe tener fecha igual o anterior al primer día de los datos.
# Mientras TASAS_DE_RELLENO sea True, las tasas son CONSTANTES: USD y MXN son los valores actuales (Fed y Banxico) y las demás son 0.0
# solo para que el código corra. Con eso el carry histórico queda mal calculado y los resultados NO son publicables.
TASAS_DE_RELLENO = True
tasas = pd.DataFrame(
    {"USD": [3.875, 3.875],      # Fed: punto medio del rango de 3.75% a 4.00%
     "MXN": [6.25, 6.25],        # Banxico
     "JPY": [0.0, 0.0], "EUR": [0.0, 0.0], "GBP": [0.0, 0.0],      # relleno, NO son las tasas reales
     "CAD": [0.0, 0.0], "BRL": [0.0, 0.0]},                        # relleno, NO son las tasas reales
    index=pd.to_datetime(["2020-10-07", "2026-11-13"]))
if TASAS_DE_RELLENO:
    print("\nAVISO: tasas de relleno constantes. Resultados solo para probar el código, NO para publicar.")
tasas = tasas.reindex(tasas.index.union(rend.index)).ffill().loc[rend.index]
if tasas.isna().any().any():
    raise ValueError("Faltan tasas al inicio: agrega una fila con fecha igual o anterior al primer día de los datos")
carry = pd.DataFrame({par: (tasas["USD"] - tasas[mon]) / 100 / 252 for par, mon in mapa_moneda.items()})
rend_total = rend + carry          # rendimiento del precio más el carry diario de la posición
resumen = pd.DataFrame({"precio_%_anual": rend.mean() * 252 * 100,
                        "carry_%_anual": carry.mean() * 252 * 100,
                        "total_%_anual": rend_total.mean() * 252 * 100})
print(resumen.round(2).to_string())


AVISO: tasas de relleno constantes. Resultados solo para probar el código, NO para publicar.
         precio_%_anual  carry_%_anual  total_%_anual
USD/MXN           -2.04          -2.38          -4.41
USD/JPY            6.91           3.88          10.79
USD/EUR            0.98           3.88           4.85
USD/GBP           -0.26           3.88           3.61
USD/CAD            1.07           3.88           4.95
USD/BRL           -0.69           3.88           3.18


In [7]:
# 7. ¿Cómo elegir el mejor lambda? Tres opciones, todas sin mirar el periodo de prueba
def encoger_jorion(mu_hat, cov, T_anios):
    n = len(mu_hat)
    inv = np.linalg.inv(cov)
    uno = np.ones(n)
    mu_g = (uno @ inv @ mu_hat) / (uno @ inv @ uno)        # rendimiento de la cartera de mínima varianza
    d = mu_hat - mu_g * uno
    w = (n + 2) / ((n + 2) + T_anios * (d @ inv @ d))      # peso que se le da al valor central
    return (1 - w) * mu_hat + w * mu_g * uno, w

def cartera_lambda(datos, lam):
    mu = datos.mean().values * 252
    cov = LedoitWolf().fit(datos.values).covariance_ * 252
    return max_sharpe_solo_largo(encoger_mu(mu, lam), cov, avisar=False)

def cartera_jorion(datos):
    mu = datos.mean().values * 252
    cov = LedoitWolf().fit(datos.values).covariance_ * 252
    mu_s, _ = encoger_jorion(mu, cov, len(datos) / 252)
    return max_sharpe_solo_largo(mu_s, cov, avisar=False)

def sharpe_serie(x):
    x = np.asarray(x)
    return float(x.mean() / x.std() * np.sqrt(252))

def elegir_lambda(datos, malla, inicio_interno=252, paso=63):
    """Validación hacia adelante DENTRO del entrenamiento: puntúa cada lambda con tramos que no usó para estimar."""
    piezas = {lam: [] for lam in malla}
    for ini in range(inicio_interno, len(datos) - paso + 1, paso):
        tr, te = datos.iloc[:ini], datos.iloc[ini:ini + paso]
        for lam in malla:
            piezas[lam].append(te.values @ cartera_lambda(tr, lam))
    sh = {lam: sharpe_serie(np.concatenate(piezas[lam])) for lam in malla}
    anios = len(np.concatenate(piezas[malla[0]])) / 252
    umbral = max(sh.values()) - 1 / np.sqrt(anios)          # regla de un error estándar: el lambda más encogido que no se queda atrás
    return min(lam for lam in malla if sh[lam] >= umbral), sh

malla = [0.0, 0.25, 0.5, 0.75, 1.0]
entren_min, paso = 756, 63
fuera = {"Lambda elegido (anidado)": [], "Jorion (Bayes Stein)": [], "Pesos iguales": []}
for lam in malla:
    fuera[f"Lambda fijo {lam}"] = []
elegidos = []
for ini in range(entren_min, len(rend_total) - paso + 1, paso):
    tr, te = rend_total.iloc[:ini], rend_total.iloc[ini:ini + paso]
    lam_el, _ = elegir_lambda(tr, malla)
    elegidos.append(lam_el)
    fuera["Lambda elegido (anidado)"].append(te.values @ cartera_lambda(tr, lam_el))
    fuera["Jorion (Bayes Stein)"].append(te.values @ cartera_jorion(tr))
    fuera["Pesos iguales"].append(te.values @ (np.ones(tr.shape[1]) / tr.shape[1]))
    for lam in malla:
        fuera[f"Lambda fijo {lam}"].append(te.values @ cartera_lambda(tr, lam))
n_dias = len(np.concatenate(fuera["Pesos iguales"]))
print("\nPliegues:", len(elegidos), "| días fuera de muestra:", n_dias, f"({n_dias / 252:.1f} años)",
      "| error estándar aproximado del Sharpe:", round(1 / np.sqrt(n_dias / 252), 2))
print("Lambda elegido en cada pliegue:", elegidos)
for nombre, piezas in fuera.items():
    serie = np.concatenate(piezas)
    print(f"  {nombre}: Sharpe {sharpe_serie(serie):.2f} | rendimiento {serie.mean() * 252 * 100:.2f}% | volatilidad {serie.std() * np.sqrt(252) * 100:.2f}%")


Pliegues: 12 | días fuera de muestra: 756 (3.0 años) | error estándar aproximado del Sharpe: 0.58
Lambda elegido en cada pliegue: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Lambda elegido (anidado): Sharpe 0.86 | rendimiento 4.61% | volatilidad 5.38%
  Jorion (Bayes Stein): Sharpe 0.91 | rendimiento 5.74% | volatilidad 6.28%
  Pesos iguales: Sharpe 0.62 | rendimiento 3.74% | volatilidad 6.06%
  Lambda fijo 0.0: Sharpe 0.86 | rendimiento 4.61% | volatilidad 5.38%
  Lambda fijo 0.25: Sharpe 0.94 | rendimiento 5.47% | volatilidad 5.84%
  Lambda fijo 0.5: Sharpe 0.93 | rendimiento 5.91% | volatilidad 6.38%
  Lambda fijo 0.75: Sharpe 0.90 | rendimiento 6.17% | volatilidad 6.85%
  Lambda fijo 1.0: Sharpe 0.87 | rendimiento 6.31% | volatilidad 7.27%
